# Ejercicio 3 — ¿Podemos lanzar el experimento?

In [1]:
from sympy import symbols, And, Or, Not, Implies, Equivalent
from sympy.logic.inference import satisfiable


def consultar(base, proposicion):
    if satisfiable(base) is False:
        return "Base inconsistente"
    if satisfiable(And(base, Not(proposicion))) is False:
        return "Verdadero"
    if satisfiable(And(base, proposicion)) is False:
        return "Falso"
    return "Desconocido"

## 1. Modelar el protocolo

**Vocabulario.** Siete observaciones (lo que se responde en la tabla) y nueve
proposiciones derivadas (lo que el protocolo concluye).

In [2]:
# Observaciones
DatosValidados, UsoAutorizado = symbols("DatosValidados UsoAutorizado")
Dependencias, PruebaMinima = symbols("Dependencias PruebaMinima")
GPUDisponible, AdmiteCPU, CPUDisponible = symbols("GPUDisponible AdmiteCPU CPUDisponible")

# Conclusiones
DatosListos, EntornoListo, Preparado = symbols("DatosListos EntornoListo Preparado")
LanzarGPU, LanzarCPU, Iniciar = symbols("LanzarGPU LanzarCPU Iniciar")
ValidarDatos, GestionarAutorizacion, RevisarEntorno = symbols(
    "ValidarDatos GestionarAutorizacion RevisarEntorno")

**Reglas.** Las dos primeras son definiciones: el protocolo dice que «datos
listos» es *exactamente* validados y autorizados, y lo mismo para el entorno.
Eso es un bicondicional (`Equivalent`): si falta una condición, los datos **no**
están listos, y eso se puede afirmar. Las demás dicen «si … entonces …» y van
con `Implies`: dicen cuándo se concluye algo, no cuándo se descarta.

In [3]:
reglas = [
    Equivalent(DatosListos, And(DatosValidados, UsoAutorizado)),
    Equivalent(EntornoListo, And(Dependencias, PruebaMinima)),
    Implies(And(DatosListos, EntornoListo), Preparado),
    Implies(And(Preparado, GPUDisponible), LanzarGPU),
    Implies(And(Preparado, AdmiteCPU, CPUDisponible), LanzarCPU),
    Implies(Or(LanzarGPU, LanzarCPU), Iniciar),
    Implies(Not(DatosValidados), ValidarDatos),
    Implies(Not(UsoAutorizado), GestionarAutorizacion),
    Implies(Not(PruebaMinima), RevisarEntorno),
]
protocolo = And(*reglas)

## 2. Los tres casos

Cada caso es una base distinta: el protocolo más las observaciones de ese
caso. El `?` del caso C no se escribe: no agregar nada es no saber.

In [4]:
caso_A = [DatosValidados, UsoAutorizado, Dependencias, PruebaMinima,
          GPUDisponible, Not(AdmiteCPU), Not(CPUDisponible)]
caso_B = [DatosValidados, UsoAutorizado, Dependencias, PruebaMinima,
          Not(GPUDisponible), AdmiteCPU, CPUDisponible]
caso_C = [DatosValidados, Dependencias, PruebaMinima,
          GPUDisponible, Not(AdmiteCPU), Not(CPUDisponible)]

preguntas = [DatosListos, EntornoListo, Preparado, LanzarGPU, LanzarCPU, Iniciar]


def informar(observaciones):
    base = And(protocolo, *observaciones)
    for p in preguntas:
        print(f"  {str(p):>13}: {consultar(base, p)}")


print("Caso A"); informar(caso_A)
print("Caso B"); informar(caso_B)
print("Caso C"); informar(caso_C)

Caso A
    DatosListos: Verdadero
   EntornoListo: Verdadero
      Preparado: Verdadero
      LanzarGPU: Verdadero
      LanzarCPU: Desconocido
        Iniciar: Verdadero
Caso B
    DatosListos: Verdadero
   EntornoListo: Verdadero
      Preparado: Verdadero
      LanzarGPU: Desconocido
      LanzarCPU: Verdadero
        Iniciar: Verdadero
Caso C
    DatosListos: Desconocido
   EntornoListo: Verdadero
      Preparado: Desconocido
      LanzarGPU: Desconocido
      LanzarCPU: Desconocido
        Iniciar: Desconocido


**Caso A.** Datos y entorno listos por las dos definiciones; preparado por la
regla 3; GPU disponible, así que se puede lanzar en GPU y se recomienda
iniciar. `LanzarCPU` queda **Desconocido**, no Falso: la regla 5 dice cuándo
se puede lanzar en CPU, pero ninguna regla dice cuándo *no* se puede. Que el
experimento no admita CPU no es una contradicción, es una condición que no se
cumple.

**Caso B.** Simétrico: sin GPU, pero admite CPU y hay una, así que
`LanzarCPU` Verdadero e `Iniciar` Verdadero. `LanzarGPU` Desconocido por la
misma razón.

**Caso C.** Falta la autorización. `DatosListos` es Desconocido (una de las dos
condiciones de la definición no se sabe), y de ahí en adelante nada se puede
concluir: `Preparado`, `LanzarGPU` e `Iniciar` quedan Desconocidos aunque haya
GPU. Tampoco se recomienda gestionar la autorización: la regla 8 se dispara
cuando se *comprobó* que falta, no cuando no se sabe.

## 3. Caso C con y sin autorización, en dos bases separadas

In [5]:
caso_C_si = caso_C + [UsoAutorizado]
caso_C_no = caso_C + [Not(UsoAutorizado)]

recomendaciones = [ValidarDatos, GestionarAutorizacion, RevisarEntorno]

for nombre, observaciones in [("C con autorización", caso_C_si), ("C sin autorización", caso_C_no)]:
    base = And(protocolo, *observaciones)
    print(nombre)
    for p in preguntas + recomendaciones:
        print(f"  {str(p):>21}: {consultar(base, p)}")

C con autorización
            DatosListos: Verdadero
           EntornoListo: Verdadero
              Preparado: Verdadero
              LanzarGPU: Verdadero
              LanzarCPU: Desconocido
                Iniciar: Verdadero
           ValidarDatos: Desconocido
  GestionarAutorizacion: Desconocido
         RevisarEntorno: Desconocido
C sin autorización


            DatosListos: Falso
           EntornoListo: Verdadero
              Preparado: Desconocido
              LanzarGPU: Desconocido
              LanzarCPU: Desconocido
                Iniciar: Desconocido
           ValidarDatos: Desconocido
  GestionarAutorizacion: Verdadero
         RevisarEntorno: Desconocido


Con autorización, C se vuelve igual a A: preparado, GPU, iniciar.

Sin autorización aparece la utilidad del bicondicional: `DatosListos` es
**Falso**, no Desconocido, porque la definición dice que los datos están listos
*exactamente cuando* validados y autorizados. Y se dispara la recomendación
`GestionarAutorizacion`.

Pero fijate que `Preparado`, `LanzarGPU` e `Iniciar` siguen **Desconocidos**,
no Falsos. La regla 3 dice que si los datos están listos, está preparado; no
dice que si no lo están, no lo está. Para poder negar `Preparado` haría falta
otro bicondicional. Es la misma lección del ejercicio 1: que no se active una
regla no niega su conclusión.

## 4. Un caso inconsistente

Una contradicción es afirmar algo y su negación, no una condición que no se
cumple. Por ejemplo: alguien registra que los datos están listos, y otra
persona registra que no fueron validados.

In [6]:
caso_malo = [DatosListos, Not(DatosValidados), Dependencias, PruebaMinima, GPUDisponible]
base = And(protocolo, *caso_malo)

print("Iniciar:     ", consultar(base, Iniciar))
print("ValidarDatos:", consultar(base, ValidarDatos))

Iniciar:      Base inconsistente
ValidarDatos: Base inconsistente


`consultar` verifica la consistencia antes que nada y responde
**Base inconsistente** a cualquier pregunta. Sin esa verificación, de una base
contradictoria se demuestra cualquier cosa: `Iniciar` y `ValidarDatos` saldrían
Verdaderos a la vez.

En cambio el caso A con `Not(AdmiteCPU)` **no** es inconsistente: el
experimento no puede correr en CPU, pero la base tiene modelos. Una
imposibilidad de ejecución es una respuesta Desconocido o Falso; una
contradicción lógica es una base sin modelos.